# Phase 3b — Predicting recommendations with Spark MLlib

Phases 2 and 3a established *that* playtime tracks the recommendation rate.
This notebook turns the question into a supervised learning problem: given
what we know about a review and the game it belongs to, can we predict
whether the player recommended it — and which of those signals actually
carries the weight?

Requires HDFS to be running (`start-dfs.sh`, `start-yarn.sh`).

In [1]:
import getpass

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
USER = getpass.getuser()
HDFS_BASE = f"hdfs://localhost:9000/user/{USER}/steam"
INPUT_PATH = f"{HDFS_BASE}/streaming_input/*.csv"

spark = (
    SparkSession.builder
    .appName("steam-reviews-ml")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

26/09/08 21:30:03 WARN Utils: Your hostname, luca-Katana-15-B13VFK resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlo1)
26/09/08 21:30:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/08 21:30:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Features

Five predictors, deliberately few: the numeric ones are taken as they are,
and the price bucket is one-hot encoded. `is_recommended` becomes the label.

`positive_ratio` is the share of positive reviews the game has on the store,
so it describes the game's reputation rather than this particular player's
experience.

In [3]:
raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)

data = (
    raw
    .withColumn("year", F.year("date"))
    .withColumn("label", F.col("is_recommended").cast("int"))
    .select("hours", "price_final", "positive_ratio", "year", "price_bucket", "label")
    .dropna()
)

print(f"rows: {data.count():,}")
data.show(5)

rows: 500,000
+-----+-----------+--------------+----+------------+-----+
|hours|price_final|positive_ratio|year|price_bucket|label|
+-----+-----------+--------------+----+------------+-----+
|558.5|       20.0|            83|2020|         mid|    1|
| 52.7|       20.0|            89|2020|         mid|    1|
|823.5|       15.0|            88|2020|         mid|    1|
|614.7|       15.0|            88|2020|         mid|    1|
| 10.0|       20.0|            83|2020|         mid|    0|
+-----+-----------+--------------+----+------------+-----+
only showing top 5 rows



In [4]:
# Class balance: worth knowing before reading any accuracy figure
data.groupBy("label").count().withColumn(
    "share", F.round(F.col("count") / data.count(), 4)
).show()

[Stage 11:====================================================>   (16 + 1) / 17]

+-----+------+------+
|label| count| share|
+-----+------+------+
|    1|422623|0.8452|
|    0| 77377|0.1548|
+-----+------+------+



In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

NUMERIC = ["hours", "price_final", "positive_ratio", "year"]

indexer = StringIndexer(inputCol="price_bucket", outputCol="price_idx")
encoder = OneHotEncoder(inputCols=["price_idx"], outputCols=["price_vec"])
assembler = VectorAssembler(inputCols=NUMERIC + ["price_vec"], outputCol="features")

prep = Pipeline(stages=[indexer, encoder, assembler]).fit(data)
prepared = prep.transform(data).select("features", "label").cache()

prepared.show(3, truncate=False)

[Stage 17:===>                                                    (1 + 16) / 17]

+------------------------------------+-----+
|features                            |label|
+------------------------------------+-----+
|[558.5,20.0,83.0,2020.0,0.0,1.0,0.0]|1    |
|[52.7,20.0,89.0,2020.0,0.0,1.0,0.0] |1    |
|[823.5,15.0,88.0,2020.0,0.0,1.0,0.0]|1    |
+------------------------------------+-----+
only showing top 3 rows



## Train / test split

A fixed seed keeps the split reproducible across runs.

In [6]:
train, test = prepared.randomSplit([0.8, 0.2], seed=42)

print(f"train: {train.count():,}")
print(f"test:  {test.count():,}")

train: 400,336
test:  99,664
